# Hafta 7 · Derin Öğrenme: Ağların Eğitimi

**Yapay Zekâya Giriş — Laboratuvar Defteri**  
Dr. Seda Bayat Toksöz · Iğdır Üniversitesi · 2026–2027 Güz

Bu defterde notlardaki çözümlü örnekleri PyTorch'un otomatik türeviyle doğrulayacak ve bir MLP'yi eğiteceksiniz. Hızlı çalışması için eğitim örneği scikit-learn'ün yerleşik 8×8 el yazısı rakam verisiyle (load_digits) yapılmıştır; notlardaki MNIST sürümü defterin sonunda ayrı bir hücre olarak verilmiştir (internet bağlantısı gerektirir).

Hücreleri yukarıdan aşağıya sırayla çalıştırın (**Shift + Enter**). Parametreleri değiştirip sonucun nasıl değiştiğini gözlemleyin.

In [1]:
# Grafik ayarları (ders notlarıyla aynı renk paleti)
import matplotlib.pyplot as plt
plt.rcParams.update({
    "figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.prop_cycle": plt.cycler(color=["#0072B2", "#D55E00", "#009E73", "#E69F00", "#56B4E9", "#CC79A7"]),
})

## 1. Softmax ve çapraz entropi (Çözümlü Örnek 7.1)

In [2]:
import torch
from torch import nn

z = torch.tensor([2.0, 1.0, 0.1])   # logitler: kedi, köpek, kuş
p = torch.softmax(z, dim=0)
print("olasılıklar:", p.numpy().round(3))

kayip_fn = nn.CrossEntropyLoss()    # softmax + çapraz entropi
for ad, sinif in [("kedi", 0), ("kuş", 2)]:
    L = kayip_fn(z.unsqueeze(0), torch.tensor([sinif]))
    print(f"gerçek sınıf {ad}: kayıp = {L.item():.3f}")

olasılıklar: [0.659 0.242 0.099]
gerçek sınıf kedi: kayıp = 0.417
gerçek sınıf kuş: kayıp = 2.317


## 2. Otomatik türev: hesaplama grafiği (Çözümlü Örnek 7.3)
PyTorch ileri geçişte grafiği kaydeder; `backward()` zincir kuralını çıkıştan girdiye uygular.

In [3]:
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)
z = torch.tensor(-4.0, requires_grad=True)

q = x + y            # toplama düğümü
f = q * z            # çarpma düğümü
f.backward()         # geri geçiş: zincir kuralı
print("f =", f.item())
print("df/dx =", x.grad.item(), " df/dy =", y.grad.item(),
      " df/dz =", z.grad.item())

f = -20.0
df/dx = -4.0  df/dy = -4.0  df/dz = 5.0


## 3. Geri yayılımı doğrulamak (Çözümlü Örnek 7.4)
6. haftadaki ağ; gerçek etiket y = 1, ikili çapraz entropi, η = 0.1.

In [4]:
T = lambda v: torch.tensor(v, requires_grad=True)
W1, b1 = T([[-0.5, 0.1], [0.2, 0.8]]), T([0.1, -0.2])
W2, b2 = T([[0.7, 0.5]]), T([-0.3])
x, y = torch.tensor([1.0, 2.0]), torch.tensor([1.0])

def ileri():
    h = torch.relu(W1 @ x + b1)          # gizli katman
    return torch.sigmoid(W2 @ h + b2)    # çıkış: y_hat

bce = nn.BCELoss()                       # ikili çapraz entropi
L = bce(ileri(), y)
L.backward()                             # tüm gradyanlar
yaz = lambda t: (t.double() + 0.0).numpy().round(3).tolist()
print("kayıp:", round(L.item(), 3))
print("dL/dW2:", yaz(W2.grad), " dL/db2:", yaz(b2.grad))
print("dL/dW1:", yaz(W1.grad))
print("dL/db1:", yaz(b1.grad))

with torch.no_grad():                    # η = 0.1 ile güncelleme
    for p in (W1, b1, W2, b2):
        p -= 0.1 * p.grad
print("yeni kayıp:", round(bce(ileri(), y).item(), 3))

kayıp: 0.474
dL/dW2: [[0.0, -0.604]]  dL/db2: [-0.378]
dL/dW1: [[0.0, 0.0], [-0.189, -0.378]]
dL/db1: [0.0, -0.189]
yeni kayıp: 0.404


## 4. Adam'ın adımları (İleri Düzey kutusundaki tablo)
Gradyanları elle veriyoruz: g = 2, 1, −0.5; η = 0.1, β₁ = 0.9, β₂ = 0.999.

In [5]:
w = torch.zeros(1, requires_grad=True)
opt = torch.optim.Adam([w], lr=0.1, betas=(0.9, 0.999))
for t, g in enumerate([2.0, 1.0, -0.5], start=1):
    once = w.item()
    w.grad = torch.tensor([g])             # gradyanı elle veriyoruz
    opt.step()
    print(f"t={t}: g={g:+.1f}  Adam adımı = {once - w.item():.3f}"
          f"  SGD adımı = {0.1 * g:.3f}")

t=1: g=+2.0  Adam adımı = 0.100  SGD adımı = 0.200
t=2: g=+1.0  Adam adımı = 0.093  SGD adımı = 0.100
t=3: g=-0.5  Adam adımı = 0.056  SGD adımı = -0.050


## 5. Toplu normalizasyon (Çözümlü Örnek 7.5)

In [6]:
bn = nn.BatchNorm1d(1, eps=1e-8)           # tek öznitelik; ϵ ≈ 0
with torch.no_grad():
    bn.weight.fill_(2.0); bn.bias.fill_(1.0)   # γ = 2, β = 1
x = torch.tensor([[2.0], [4.0], [6.0], [8.0]]) # 4 örneklik yığın
bn.train()                                 # eğitimde yığın istatistikleri
print(bn(x).detach().ravel().numpy().round(3))

[-1.683  0.106  1.894  3.683]


### Alıştırma (Değerlendirme Sorusu 8)
Logitler z = [3, 1, 0]; doğru sınıf ikinci sınıf. Önce elle hesaplayın, sonra çalıştırın.

In [ ]:
z2 = torch.tensor([3.0, 1.0, 0.0])
print(torch.softmax(z2, dim=0).numpy().round(3))
print(round(nn.CrossEntropyLoss()(z2.unsqueeze(0), torch.tensor([1])).item(), 3))

## 6. PyTorch ile bir MLP eğitmek
Veri: scikit-learn'ün yerleşik **8×8 el yazısı rakamlar** (1 797 görüntü, 10 sınıf). İnternet gerektirmez ve CPU'da saniyeler içinde eğitilir.

In [8]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(0)
X, y = load_digits(return_X_y=True)      # 1 797 görüntü, 8×8 piksel
X_eg, X_te, y_eg, y_te = train_test_split(
    X / 16.0, y, test_size=0.2, random_state=0, stratify=y)  # [0, 1]
T = lambda a: torch.tensor(a, dtype=torch.float32).view(-1, 1, 8, 8)
egitim = TensorDataset(T(X_eg), torch.tensor(y_eg))
test = TensorDataset(T(X_te), torch.tensor(y_te))
egitim_yukleyici = DataLoader(egitim, batch_size=64, shuffle=True)
test_yukleyici = DataLoader(test, batch_size=256)
print(len(egitim), "eğitim,", len(test), "test örneği")
print(len(egitim_yukleyici), "iterasyon/epok")

1437 eğitim, 360 test örneği
23 iterasyon/epok


In [9]:
model = nn.Sequential(
    nn.Flatten(),                        # 1x8x8 -> 64
    nn.Linear(64, 128), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(128, 64), nn.ReLU(),
    nn.Linear(64, 10))                   # 10 logit
kayip_fn = nn.CrossEntropyLoss()         # softmax + kategorik ÇE
optimizator = torch.optim.Adam(model.parameters(), lr=1e-3)
print(model)
print("parametre:", sum(p.numel() for p in model.parameters()))

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=64, out_features=128, bias=True)
  (2): ReLU()
  (3): Dropout(p=0.2, inplace=False)
  (4): Linear(in_features=128, out_features=64, bias=True)
  (5): ReLU()
  (6): Linear(in_features=64, out_features=10, bias=True)
)
parametre: 17226


### Eğitim döngüsü: ileri yayılım → kayıp → geri yayılım → güncelleme

In [10]:
for epok in range(20):
    model.train()                          # dropout açık
    for x, y in egitim_yukleyici:
        y_hat = model(x)                   # 1) ileri yayılım
        kayip = kayip_fn(y_hat, y)         # 2) kayıp
        optimizator.zero_grad()            #    eski gradyanları sıfırla
        kayip.backward()                   # 3) geri yayılım
        optimizator.step()                 # 4) güncelleme
    model.eval()                           # dropout kapalı
    dogru = 0
    with torch.no_grad():                  # testte gradyan gerekmez
        for x, y in test_yukleyici:
            dogru += (model(x).argmax(dim=1) == y).sum().item()
    if (epok + 1) % 4 == 0:              # her 4 epokta bir yazdır
        print(f"epok {epok+1}: doğruluk = {dogru/len(test):.4f}")

epok 4: doğruluk = 0.8694


epok 8: doğruluk = 0.9278


epok 12: doğruluk = 0.9417


epok 16: doğruluk = 0.9667


epok 20: doğruluk = 0.9639


## 7. Tam sürüm: MNIST (ders notlarındaki kod)
Bu hücre MNIST veri setini (60 000 eğitim, 10 000 test görüntüsü, 28×28) internetten indirir; Colab'da çalıştırın. Model notlardaki 784–128–64–10 MLP'dir; 5 epok sonunda test doğruluğu tipik olarak %97–98 civarındadır.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

donusum = transforms.ToTensor()            # pikselleri [0, 1] aralığına getirir
egitim = datasets.MNIST("veri", train=True, download=True, transform=donusum)
test = datasets.MNIST("veri", train=False, download=True, transform=donusum)
egitim_yukleyici = DataLoader(egitim, batch_size=64, shuffle=True)
test_yukleyici = DataLoader(test, batch_size=256)

model = nn.Sequential(
    nn.Flatten(),                          # 1x28x28 -> 784
    nn.Linear(784, 128), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(128, 64), nn.ReLU(),
    nn.Linear(64, 10))                     # 10 logit (softmax kaybın içinde)
kayip_fn = nn.CrossEntropyLoss()           # softmax + kategorik çapraz entropi
optimizator = torch.optim.Adam(model.parameters(), lr=1e-3)

for epok in range(5):
    model.train()                          # dropout açık
    for x, y in egitim_yukleyici:
        y_hat = model(x)                   # 1) ileri yayılım
        kayip = kayip_fn(y_hat, y)         # 2) kayıp
        optimizator.zero_grad()            #    eski gradyanları sıfırla
        kayip.backward()                   # 3) geri yayılım
        optimizator.step()                 # 4) güncelleme
    model.eval()                           # dropout kapalı
    dogru = 0
    with torch.no_grad():                  # testte gradyan gerekmez
        for x, y in test_yukleyici:
            dogru += (model(x).argmax(dim=1) == y).sum().item()
    print(f"epok {epok+1}: test doğruluğu = {dogru / len(test):.4f}")
# 5 epok sonunda test doğruluğu tipik olarak %97-98 civarındadır.